In [ ]:
%time 


import geopandas as gpd
from shapely.geometry import Point, polygon, LineString

from shapely.geometry import Point, Polygon, LineString
from shapely import geometry
from shapely.geometry import MultiPolygon, Polygon, LineString, Point, MultiPoint
import IPython.display as display
#import contextily as ctx
#import mplleaflet
#import geoplot.crs as gcrs

import matplotlib.pyplot as plt
import seaborn as sb
import pandas as pd
import numpy as np
from datetime import datetime
import ipywidgets as widgets
import copy
import warnings
warnings.filterwarnings('ignore')

#from bioinfokit import analys, visuz

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [ ]:

RUTA = r"C:\Users\valeria.ulloa\Documents\trabajo_varios\DIPLOMADO_ANALISIS_ESPACIAL\diplomado_2026\segunda_parte\sinistros_chile.csv"

df = pd.read_csv(RUTA)

df_rm = df[df["REGION"] == "Metropolitana de Santiago"].copy()

df_rm.to_csv(r"\siniestros_rm.csv", index=False)


print("RM:", df_rm.shape, " Araucanía:", df_arau.shape)

##

In [ ]:
df["REGION"].unique()

In [ ]:
# De aquí en adelante trabajamos solo con RM (partimos de su CSV, como harán ustedes)
RUTA_RM = r"C:\Users\valeria.ulloa\Documents\trabajo_varios\DIPLOMADO_ANALISIS_ESPACIAL\diplomado_2026\segunda_parte\siniestros_rm.csv"

df_rm = pd.read_csv(RUTA_RM, parse_dates=['FECHA'])
print(df_rm.shape)
df_rm.head(3)

In [ ]:
df_rm.dtypes

In [ ]:
#si tenemos la direccion de un accidente o de un hecho en particular 

In [ ]:
df_rm["TIPO_SINIE"].unique()

In [ ]:
# Limpieza: descartar columnas redundantes o confirmadas como duplicadas

df_rm = df_rm.drop(columns=['SHAPE', 'Siniestro'], errors='ignore')

# Nulos y duplicados (ya sabemos que en la base completa no había, pero se revisa siempre
# después de filtrar, por si el filtro introdujo algo raro)
print("Nulos:\n", df_rm.isnull().sum()[df_rm.isnull().sum() > 0])
print("\nDuplicados:", df_rm.duplicated().sum())
print("ID duplicados:", df_rm['ID'].duplicated().sum())

In [ ]:
# Validar coordenadas (RM no debería tener puntos insulares ni de otras regiones)
print("Puntos con longitud fuera de Chile continental:", (df_rm['longitud'] < -76).sum())
print("\nLatitud:\n", df_rm['Latitud'].describe())
print("\nlongitud:\n", df_rm['longitud'].describe())

# El describe() de arriba muestra un mínimo de Latitud/longitud muy alejado del resto (la RM
# real está aprox. entre -34.4/-32.8 de latitud y -71.7/-70.0 de longitud): son siniestros con
# comuna de la RM pero coordenadas que caen en el sur de Chile (error de digitación de
# coordenadas, no de comuna). No son insulares (por eso el chequeo de arriba no los detecta),
# así que se filtran con una caja acotada a la RM real, con margen.
fuera_de_rm = ~df_rm['Latitud'].between(-34.6, -32.7) | ~df_rm['longitud'].between(-71.9, -69.9)
print(f"\nSiniestros con coordenadas fuera del rango esperado para la RM: {fuera_de_rm.sum()}")
print(df_rm.loc[fuera_de_rm, ['ID', 'COMUNA', 'Latitud', 'longitud']])

df_rm = df_rm[~fuera_de_rm].copy()
print(f"\ndf_rm después de descartar esos puntos: {df_rm.shape}")

In [ ]:
# siniestros, fallecidos y severidad por año
resumen_anual = df_rm.groupby('AÑO').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum'),
    lesionados=('TOTAL_LESI', 'sum')
)
resumen_anual['severidad'] = (resumen_anual['fallecidos'] / resumen_anual['siniestros'] * 100).round(2)
resumen_anual

In [ ]:
# Validación contra el Informe Nacional 2025 de CONASET (Tabla 5, solo RM)
REF_RM_2025 = {'siniestros': 20291, 'fallecidos': 350}
d25 = df_rm[df_rm['AÑO'] == 2025]

dif_siniestros = (len(d25) / REF_RM_2025['siniestros'] - 1) * 100
dif_fallecidos = (d25['FALLECIDOS'].sum() / REF_RM_2025['fallecidos'] - 1) * 100

print(f"Mío 2025: {len(d25)} siniestros, {d25['FALLECIDOS'].sum()} fallecidos")
print(f"CONASET 2025: {REF_RM_2025['siniestros']} siniestros, {REF_RM_2025['fallecidos']} fallecidos")
print(f"Diferencia: {dif_siniestros:.1f}% en siniestros, {dif_fallecidos:.1f}% en fallecidos")
# Nota: esta base subestima de forma consistente al informe oficial (visto también en 2024).
# Es una limitación a documentar, no un error a "corregir".

In [ ]:
#Tipo de siniestro: volumen y severidad
por_tipo = df_rm.groupby('TIPO__CONA').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum')
)
por_tipo['severidad'] = (por_tipo['fallecidos'] / por_tipo['siniestros'] * 100).round(2)
por_tipo.sort_values('siniestros', ascending=False)

In [ ]:
# Causa (reclasificada 2025): top causas por fallecidos
por_causa = df_rm.groupby('CAUSA_NUEV')['FALLECIDOS'].sum().sort_values(ascending=False)
por_causa.head(10)

In [ ]:
# Zona urbana/rural: volumen y severidad
por_zona = df_rm.groupby('ZONA').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum')
)
por_zona['severidad'] = (por_zona['fallecidos'] / por_zona['siniestros'] * 100).round(2)
por_zona

In [ ]:
# Línea temporal: fallecidos por mes
fall_mensual = df_rm.set_index('FECHA').resample('ME')['FALLECIDOS'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
fall_mensual.plot(ax=ax, color='#C8401A')
ax.set_title('RM: fallecidos por mes, 2020-2025')
ax.set_xlabel(''); ax.set_ylabel('fallecidos')
plt.tight_layout(); plt.show()

In [ ]:
# Tipo de siniestro en el tiempo (por año, para que se lea bien)
tipo_anual = df_rm.groupby(['AÑO', 'TIPO__CONA']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
tipo_anual.plot(ax=ax, marker='o')
ax.set_title('RM: siniestros por tipo, cada año')
ax.set_xlabel('año'); ax.set_ylabel('siniestros')
ax.legend(title='Tipo', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
#Construir la capa de puntos (GeoDataFrame) desde Latitud/longitud
gdf_rm = gpd.GeoDataFrame(
    df_rm,
    geometry=gpd.points_from_xy(df_rm['longitud'], df_rm['Latitud']),
    crs='EPSG:4326'   # grados, WGS84
)

# Proyectamos a metros (UTM 19S) para poder medir distancias y áreas correctamente
gdf_rm_utm = gdf_rm.to_crs('EPSG:32719')
print(gdf_rm_utm.crs, gdf_rm_utm.shape)
gdf_rm_utm[['ID', 'COMUNA', 'TIPO__CONA', 'FALLECIDOS', 'geometry']].head(3)

In [ ]:
#Mapa rápido de los puntos, para confirmar visualmente que la geometría quedó bien
fig, ax = plt.subplots(figsize=(6, 8))
gdf_rm_utm.plot(ax=ax, markersize=0.5, alpha=0.3, color='#1D4F91')
ax.set_title(f'RM: {len(gdf_rm_utm)} siniestros')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

In [ ]:
# Cargar la capa de zonas censales del INE
RUTA_ZONAS = r"C:\Users\valeria.ulloa\Documents\cartografias\cartografia_chile_2017\zc chile\zonas_censales_chile.shp"

zonas = gpd.read_file(RUTA_ZONAS)
print("CRS de las zonas:", zonas.crs)
print("Filas:", len(zonas))
print("Columnas:", zonas.columns.tolist())
zonas.head(3)

In [ ]:
# Mirar los primeros valores de las columnas que probablemente identifican comuna/región,
# para saber cómo filtrar solo las zonas de la RM antes de cruzar (así el sjoin es mucho más rápido).
# Ajusta el nombre de columna
# como ejemplo; reemplázalos por los reales de tu shapefile).
for col in zonas.columns:
    if zonas[col].dtype == object and zonas[col].nunique() < 400:
        print(f"\n{col} (primeros valores únicos):")
        print(zonas[col].unique()[:15])

In [ ]:
#Filtrar las zonas censales que son realmente de la RM,
# usando el atributo de región (más preciso que el bounding box)
zonas_utm = zonas.to_crs(gdf_rm_utm.crs)

zonas_rm = zonas_utm[zonas_utm['NOM_REGION'] == 'REGIÓN METROPOLITANA DE SANTIAGO'].reset_index(drop=True)
print(f"Zonas censales de la RM: {len(zonas_rm)} (de {len(zonas_utm)} totales en Chile)")

# Chequeo cruzado: comparar contra el filtro por bounding box de antes, para detectar sorpresas
xmin, ymin, xmax, ymax = gdf_rm_utm.total_bounds
zonas_bbox = zonas_utm.cx[xmin:xmax, ymin:ymax]
print(f"(Referencia: {len(zonas_bbox)} zonas caían dentro del rectángulo de los puntos)")

In [ ]:
# Cruce espacial: qué zona censal le corresponde a cada siniestro
zonas_rm['zona_id'] = zonas_rm.index

cruce = gpd.sjoin(gdf_rm_utm, zonas_rm[['zona_id', 'geometry']], how='left', predicate='within')

sin_zona = cruce['zona_id'].isna().sum()
print(f"Siniestros que no cayeron dentro de ninguna zona censal: {sin_zona} ({sin_zona/len(cruce)*100:.1f}%)")
# Un porcentaje bajo es normal (puntos justo en el borde, o en zonas rurales sin polígono censal urbano).
# Un porcentaje alto avisaría de un problema de CRS o de que las zonas no cubren toda la RM.

In [ ]:
cruce.head(2)

In [ ]:
cruce.explore()

In [ ]:
# Agregar: siniestros y fallecidos por zona censal
conteo_por_zona = cruce.groupby('zona_id').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum')
).reset_index()

zonas_rm = zonas_rm.merge(conteo_por_zona, on='zona_id', how='left') #no es inner 
zonas_rm[['siniestros', 'fallecidos']] = zonas_rm[['siniestros', 'fallecidos']].fillna(0)

zonas_rm.sort_values('siniestros', ascending=False).head(10)

In [ ]:
conteo_por_zona.head(2)

In [ ]:
# Mapa coroplético: siniestros por zona censal
fig, ax = plt.subplots(figsize=(8, 10))
zonas_rm.plot(column='siniestros', cmap='Blues', legend=True, ax=ax,
              edgecolor='white', linewidth=0.1, missing_kwds={'color': '#EEEEEE'})
ax.set_title('RM: siniestros por zona censal (2020-2025)')
ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
# Guardar el resultado (capa con el conteo) para reutilizar en la clase o en el informe
zonas_rm.to_file(
    r"C:\Users\valeria.ulloa\Documents\trabajo_varios\DIPLOMADO_ANALISIS_ESPACIAL\diplomado_2026\segunda_parte\rm_siniestros_por_zona.gpkg",
    driver='GPKG'
)
print("Guardado.")

In [ ]:
#Ranking de comunas por número de siniestros
import plotly.express as px

por_comuna = df_rm.groupby('COMUNA').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum')
).reset_index()
por_comuna['severidad'] = (por_comuna['fallecidos'] / por_comuna['siniestros'] * 100).round(2)
por_comuna = por_comuna.sort_values('siniestros', ascending=False)

fig = px.bar(
    por_comuna.head(20),
    x='siniestros', y='COMUNA', orientation='h',
    title='RM: comunas con más siniestros (2020-2025)',
    text='siniestros', color='severidad', color_continuous_scale='Blues',
    hover_data={'fallecidos': True, 'severidad': True}
)
fig.update_yaxes(categoryorder='total ascending')
fig.update_layout(height=600)
fig.show()

In [ ]:
#La misma pregunta, pero desglosada por tipo de siniestro (barras apiladas)
# para ver no solo QUÉ comuna gana, sino CON QUÉ tipo de siniestro gana
top15 = por_comuna.head(15)['COMUNA']
detalle = df_rm[df_rm['COMUNA'].isin(top15)].groupby(['COMUNA', 'TIPO__CONA']).size().reset_index(name='siniestros')

fig = px.bar(
    detalle, x='siniestros', y='COMUNA', color='TIPO__CONA', orientation='h',
    title='RM: comunas con más siniestros, por tipo',
    category_orders={'COMUNA': list(top15)}
)
fig.update_layout(height=650, legend_title='Tipo')
fig.show()

In [ ]:
#Patrón horario: ¿a qué hora ocurren más siniestros, y cuáles son más graves?
por_hora = df_rm.groupby('Hora_aprox').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum')
)
por_hora['severidad'] = (por_hora['fallecidos'] / por_hora['siniestros'] * 100).round(2)

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(por_hora.index, por_hora['siniestros'], color='#1D4F91', alpha=0.85, label='siniestros')
ax1.set_xlabel('hora del día'); ax1.set_ylabel('siniestros', color='#1D4F91')
ax1.set_xticks(range(0, 24))

ax2 = ax1.twinx()
ax2.plot(por_hora.index, por_hora['severidad'], color='#C8401A', marker='o', linewidth=2, label='severidad')
ax2.set_ylabel('severidad (fallecidos cada 100 siniestros)', color='#C8401A')

ax1.set_title('RM: volumen y severidad de siniestros por hora del día')
plt.tight_layout(); plt.show()
# Nota: el volumen (barras) sigue el patrón de tráfico (puntas mañana/tarde), pero la severidad
# (línea) suele ser más alta en la madrugada, con menos siniestros pero más graves: la hipótesis
# habitual es velocidad más alta y menor fiscalización a esas horas.

In [ ]:
#Día de la semana × hora: ¿cambia el patrón horario según el día?
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
tabla_dia_hora = pd.crosstab(df_rm['Dia_semana'], df_rm['Hora_aprox']).reindex(orden_dias)

fig, ax = plt.subplots(figsize=(14, 5))
sb.heatmap(tabla_dia_hora, cmap='YlOrRd', linewidths=0.3, linecolor='white', ax=ax,
           cbar_kws={'label': 'siniestros'})
ax.set_title('RM: siniestros por día de la semana y hora')
ax.set_xlabel('hora del día'); ax.set_ylabel('')
plt.tight_layout(); plt.show()
# Nota: si el fin de semana (Vie-noche a Dom-madrugada) muestra un bloque caliente que entre
# semana no aparece, es un indicio de patrón asociado a vida nocturna/consumo de alcohol.

In [ ]:
#Tipo de siniestro × hora: ¿todos los tipos se concentran en las mismas horas?
tabla_tipo_hora = pd.crosstab(df_rm['TIPO__CONA'], df_rm['Hora_aprox'])
tabla_tipo_hora_pct = tabla_tipo_hora.div(tabla_tipo_hora.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 5))
sb.heatmap(tabla_tipo_hora_pct, cmap='Purples', linewidths=0.3, linecolor='white', ax=ax,
           cbar_kws={'label': '% de los siniestros de ese tipo'})
ax.set_title('RM: distribución horaria por tipo de siniestro (normalizado por fila)')
ax.set_xlabel('hora del día'); ax.set_ylabel('')
plt.tight_layout(); plt.show()
# Nota: normalizar por fila (cada tipo suma 100%) permite comparar la FORMA de la curva horaria
# entre tipos con volúmenes muy distintos (p.ej. atropello vs. colisión), en vez de que el tipo
# más frecuente (colisión) opaque visualmente a los demás.

In [ ]:
#Zona urbana/rural × hora: ¿los siniestros rurales ocurren en otro horario que los urbanos?
tabla_zona_hora = pd.crosstab(df_rm['ZONA'], df_rm['Hora_aprox'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(14, 2.5))
sb.heatmap(tabla_zona_hora, cmap='Oranges', linewidths=0.3, linecolor='white', ax=ax,
           cbar_kws={'label': '% de los siniestros de esa zona'})
ax.set_title('RM: distribución horaria, urbano vs. rural (normalizado por fila)')
ax.set_xlabel('hora del día'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
#Puntos calientes (KDE): dónde se concentran los siniestros
# Estimación de densidad de kernel (KDE) sobre las coordenadas UTM: a diferencia del mapa
# coroplético por zona censal (Celda 22), esto estima la densidad de forma continua en el
# espacio, sin depender de dónde el INE trazó los límites administrativos.
from scipy.stats import gaussian_kde

xmin, ymin, xmax, ymax = gdf_rm_utm.total_bounds

def mapa_calor_kde(x, y, titulo, ax, bins=120, cmap='inferno'):
    kde = gaussian_kde(np.vstack([x, y]))
    xx, yy = np.mgrid[xmin:xmax:complex(bins), ymin:ymax:complex(bins)]
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

    ax.contourf(xx, yy, zz, levels=25, cmap=cmap)
    zonas_rm.boundary.plot(ax=ax, linewidth=0.15, color='white', alpha=0.5)
    ax.set_title(f'{titulo}\n(n = {len(x)})')
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal'); ax.set_axis_off()

fig, ax = plt.subplots(figsize=(7, 9))
mapa_calor_kde(gdf_rm_utm.geometry.x.values, gdf_rm_utm.geometry.y.values,
               'RM: densidad de siniestros (KDE), 2020-2025', ax)
plt.tight_layout(); plt.show()

In [ ]:
#Puntos calientes por franja horaria: ¿los hotspots se mueven entre el día y la noche?
es_noche = (df_rm['Hora_aprox'] >= 20) | (df_rm['Hora_aprox'] < 6)

x_dia, y_dia = gdf_rm_utm.geometry.x.values[~es_noche], gdf_rm_utm.geometry.y.values[~es_noche]
x_noche, y_noche = gdf_rm_utm.geometry.x.values[es_noche], gdf_rm_utm.geometry.y.values[es_noche]

fig, axes = plt.subplots(1, 2, figsize=(13, 9))
mapa_calor_kde(x_dia, y_dia, 'Día (06:00-19:59)', axes[0], bins=100)
mapa_calor_kde(x_noche, y_noche, 'Noche (20:00-05:59)', axes[1], bins=100)
plt.tight_layout(); plt.show()
# Nota: si el "manchón" de mayor densidad se desplaza de un panel a otro, es señal de que la
# concentración espacial de siniestros no es fija: depende de la hora (p.ej. sectores con más
# vida nocturna vs. sectores con más congestión diurna).

In [ ]:
# Puntos calientes en folium, centrados en la RM
import folium
from folium.plugins import HeatMap

# Encuadre automático a partir de los puntos ya limpios (Celda 5), en vez de un zoom fijo a mano:
# así el mapa siempre queda centrado en la RM real, sin importar si hay outliers o no.
bounds_rm = [[df_rm['Latitud'].min(), df_rm['longitud'].min()],
             [df_rm['Latitud'].max(), df_rm['longitud'].max()]]

es_noche = (df_rm['Hora_aprox'] >= 20) | (df_rm['Hora_aprox'] < 6)

N_MUESTRA = 20000
puntos_dia = df_rm.loc[~es_noche, ['Latitud', 'longitud']].sample(
    n=min(N_MUESTRA, (~es_noche).sum()), random_state=42).values.tolist()
puntos_noche = df_rm.loc[es_noche, ['Latitud', 'longitud']].sample(
    n=min(N_MUESTRA, es_noche.sum()), random_state=42).values.tolist()

mapa_calor = folium.Map(tiles='CartoDB positron')
mapa_calor.fit_bounds(bounds_rm)

fg_dia = folium.FeatureGroup(name=f"Día 06-20h (muestra de {len(puntos_dia)})", show=True)
HeatMap(puntos_dia, radius=8, blur=6).add_to(fg_dia)
fg_dia.add_to(mapa_calor)

fg_noche = folium.FeatureGroup(name=f"Noche 20-06h (muestra de {len(puntos_noche)})", show=False)
HeatMap(puntos_noche, radius=8, blur=6).add_to(fg_noche)
fg_noche.add_to(mapa_calor)

folium.LayerControl(collapsed=False).add_to(mapa_calor)
mapa_calor

In [ ]:
# Variables para un clustering espacio-temporal (KMeans)

from sklearn.preprocessing import StandardScaler

df_kmeans = df_rm.copy()
df_kmeans['X'] = gdf_rm_utm.geometry.x.values
df_kmeans['Y'] = gdf_rm_utm.geometry.y.values
df_kmeans['hora_sin'] = np.sin(2 * np.pi * df_kmeans['Hora_aprox'] / 24)
df_kmeans['hora_cos'] = np.cos(2 * np.pi * df_kmeans['Hora_aprox'] / 24)

features = ['X', 'Y', 'hora_sin', 'hora_cos']
Xk = StandardScaler().fit_transform(df_kmeans[features])
print("Matriz de variables (espacio + hora, estandarizadas):", Xk.shape)

In [ ]:
# Celda 33 — Elegir k: método del codo (sobre una muestra, para que corra rápido)
from sklearn.cluster import KMeans

muestra_idx = np.random.RandomState(42).choice(len(Xk), size=20000, replace=False)
Xk_muestra = Xk[muestra_idx]

inercias = []
rango_k = range(2, 11)
for k in rango_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(Xk_muestra)
    inercias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(rango_k), inercias, marker='o', color='#1D4F91')
ax.set_xlabel('k (número de clusters)'); ax.set_ylabel('inercia')
ax.set_title('Método del codo (muestra de 20.000 siniestros)')
plt.tight_layout(); plt.show()
# Busca el "codo": el punto donde agregar más clusters deja de reducir mucho la inercia.

In [ ]:
#Aplicar KMeans (k elegido a partir del codo) sobre todos los siniestros
K = 6  # ajustar según el codo de la Celda 33
kmeans_final = KMeans(n_clusters=K, n_init=10, random_state=42)
df_kmeans['cluster'] = kmeans_final.fit_predict(Xk)

fig, ax = plt.subplots(figsize=(7, 9))
gpd.GeoDataFrame(df_kmeans, geometry=gdf_rm_utm.geometry).plot(
    column='cluster', categorical=True, cmap='tab10', markersize=1, alpha=0.4,
    legend=True, ax=ax
)
ax.set_title(f'RM: clusters espacio-temporales (KMeans, k={K})')
ax.set_aspect('equal'); ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
#Caracterizar cada cluster: ¿dónde está, qué tipo domina y a qué hora ocurre?
resumen_cluster = df_kmeans.groupby('cluster').agg(
    siniestros=('ID', 'size'),
    fallecidos=('FALLECIDOS', 'sum'),
    hora_media=('Hora_aprox', 'mean'),
    comuna_top=('COMUNA', lambda s: s.mode().iat[0]),
    tipo_top=('TIPO__CONA', lambda s: s.mode().iat[0]),
)
resumen_cluster['severidad'] = (resumen_cluster['fallecidos'] / resumen_cluster['siniestros'] * 100).round(2)
resumen_cluster['hora_media'] = resumen_cluster['hora_media'].round(1)
resumen_cluster.sort_values('siniestros', ascending=False)

In [ ]:
#Composición de cada cluster por tipo de siniestro
tabla_cluster_tipo = pd.crosstab(df_kmeans['cluster'], df_kmeans['TIPO__CONA'])
tabla_cluster_tipo_pct = tabla_cluster_tipo.div(tabla_cluster_tipo.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 4))
sb.heatmap(tabla_cluster_tipo_pct, cmap='Greens', annot=True, fmt='.0f', linewidths=0.3, ax=ax,
           cbar_kws={'label': '% dentro del cluster'})
ax.set_title('RM: composición por tipo de siniestro, según cluster espacio-temporal')
ax.set_xlabel('tipo de siniestro'); ax.set_ylabel('cluster')
plt.tight_layout(); plt.show()
# Si un cluster concentra desproporcionadamente un tipo (p.ej. atropello) frente al promedio
# general (Celda 8), ese cluster describe un patrón real, no solo una zona geográfica cualquiera.

In [ ]:
#DBSCAN: hotspots de densidad pura (sin fijar de antemano el número de clusters)
# A diferencia de KMeans (que reparte TODOS los puntos en k grupos, como un Voronoi), DBSCAN
# encuentra bolsones de alta densidad y deja como "ruido" (-1) los siniestros aislados. Aquí se
# aplica solo en el espacio (X, Y) para identificar puntos calientes puros, sin mezclar con la
# hora (que ya se exploró con KMeans y con el KDE por franja horaria).
from sklearn.cluster import DBSCAN

muestra_db = df_kmeans.sample(n=30000, random_state=42)
Xy_db = muestra_db[['X', 'Y']].values

# eps en metros (coordenadas en UTM): 150 m es un radio razonable para densidad urbana en la RM
dbscan = DBSCAN(eps=150, min_samples=15).fit(Xy_db)
muestra_db = muestra_db.assign(cluster_db=dbscan.labels_)

n_hotspots = muestra_db['cluster_db'].nunique() - (1 if -1 in muestra_db['cluster_db'].values else 0)
ruido_pct = (muestra_db['cluster_db'] == -1).mean() * 100
print(f"Hotspots detectados: {n_hotspots}  |  % de siniestros aislados (ruido): {ruido_pct:.1f}%")

fig, ax = plt.subplots(figsize=(7, 9))
gpd.GeoDataFrame(muestra_db, geometry=gpd.points_from_xy(muestra_db['X'], muestra_db['Y'])).plot(
    column='cluster_db', categorical=True, cmap='tab20', markersize=3, alpha=0.6, ax=ax, legend=False
)
ax.set_title(f'RM: hotspots de siniestros (DBSCAN, muestra de {len(muestra_db)})')
ax.set_aspect('equal'); ax.set_axis_off()
plt.tight_layout(); plt.show()